<a href="https://colab.research.google.com/github/rm571222/dataholics-oracle-challenge/blob/main/notebooks/01_data_exploration/nb4_data_exploration_populacao_ibge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB4 — Data Exploration: Estimativas de População (IBGE)

**Projeto DATAHOLICS — FIAP Challenge | Parceria Oracle**

Este notebook documenta a exploração da quarta e última fonte primária do projeto: as estimativas de população por município, usadas para normalizar indicadores regionais (ex.: internações por 10 mil habitantes). Esta fonte é a peça planejada para o componente de **external table (CSV)** da arquitetura final.

## Fonte de dados

- **Sistema:** Estimativas da População Residente
- **Publicação:** IBGE
- **URLs:**
  - 2024: https://ftp.ibge.gov.br/Estimativas_de_Populacao/Estimativas_2024/POP2024_20241230.xls
  - 2025: https://ftp.ibge.gov.br/Estimativas_de_Populacao/Estimativas_2025/POP2025_20260828.xls
- **Formato:** planilha Excel, com abas separadas por nível de agregação (Brasil e UFs / Municípios)

## Estrutura deste notebook

1. Exploração inicial dos dados brutos
2. Seleção das principais variáveis + tratamento
3. Exploração/qualidade dos dados tratados
4. Consistência entre os dois anos disponíveis
5. Cruzamento de validação com a fonte SIH/pysus (NB1)

## 1. Exploração inicial dos dados brutos

A planilha do IBGE não vem pronta para uso direto — tem células de título mescladas, cabeçalho fora da primeira linha, e múltiplas abas por nível de agregação. Antes de qualquer tratamento, olhamos a estrutura crua.

In [ ]:
import pandas as pd

url_2024 = "https://ftp.ibge.gov.br/Estimativas_de_Populacao/Estimativas_2024/POP2024_20241230.xls"

raw_2024 = pd.read_excel(url_2024, sheet_name=None, header=None)
print("Abas disponíveis:", list(raw_2024.keys()))
print()
print(raw_2024['MUNICÍPIOS'].head(15))

Abas disponíveis: ['BRASIL E UFs', 'MUNICÍPIOS']

                                                    0        1           2  \
0   ESTIMATIVAS DA POPULAÇÃO RESIDENTE NOS MUNICÍP...      NaN         NaN   
1                                                  UF  COD. UF  COD. MUNIC   
2                                                  RO       11       00015   
3                                                  RO       11       00023   
4                                                  RO       11       00031   
5                                                  RO       11       00049   
6                                                  RO       11       00056   
7                                                  RO       11       00064   
8                                                  RO       11       00072   
9                                                  RO       11       00080   
10                                                 RO       11       00098   
11            

A aba "BRASIL E UFs" traz totais agregados por estado; a aba de municípios traz o código dividido em duas colunas (`COD. UF` + `COD. MUNIC`), com cabeçalho começando na segunda linha do arquivo.

## 2. Seleção das principais variáveis + tratamento

O tratamento precisa: (1) pular a linha de título; (2) juntar `COD. UF` + `COD. MUNIC` no código IBGE de 7 dígitos; (3) remover o dígito verificador para chegar ao formato de 6 dígitos usado pelo SIH (`MUNIC_MOV`/`MUNIC_RES`); (4) filtrar só SP.

In [ ]:
def tratar_populacao_ibge(url, ano, nome_aba):
    raw = pd.read_excel(url, sheet_name=nome_aba, header=1)
    raw = raw.dropna(axis=1, how='all')
    raw = raw.dropna(subset=['COD. UF', 'COD. MUNIC'])  # remove notas de rodapé

    raw['COD_MUNIC_7'] = (
        raw['COD. UF'].astype(int).astype(str).str.zfill(2) +
        raw['COD. MUNIC'].astype(int).astype(str).str.zfill(5)
    )
    raw['COD_MUNIC_6'] = raw['COD_MUNIC_7'].str[:-1]

    df_sp = raw[raw['UF'] == 'SP'].copy()
    df_sp = df_sp.rename(columns={
        'NOME DO MUNICÍPIO': 'NOME_MUNICIPIO',
        'POPULAÇÃO ESTIMADA': 'POPULACAO'
    })
    df_sp['ANO_REF'] = ano

    return df_sp[['COD_MUNIC_6', 'COD_MUNIC_7', 'NOME_MUNICIPIO', 'POPULACAO', 'ANO_REF']]

df_pop_2024 = tratar_populacao_ibge(
    "https://ftp.ibge.gov.br/Estimativas_de_Populacao/Estimativas_2024/POP2024_20241230.xls",
    2024, 'MUNICÍPIOS'
)
df_pop_2025 = tratar_populacao_ibge(
    "https://ftp.ibge.gov.br/Estimativas_de_Populacao/Estimativas_2025/POP2025_20260828.xls",
    2025, 'Municípios'
)

df_populacao_sp = pd.concat([df_pop_2024, df_pop_2025], ignore_index=True)
print(f"Total de linhas (municípios x anos): {len(df_populacao_sp)}")
df_populacao_sp.head()

Total de linhas (municípios x anos): 1290


,COD_MUNIC_6,COD_MUNIC_7,NOME_MUNICIPIO,POPULACAO,ANO_REF
0,350010,3500105,Adamantina,35642.0,2024
1,350020,3500204,Adolfo,4478.0,2024
2,350030,3500303,Aguaí,32888.0,2024
3,350040,3500402,Águas da Prata,7470.0,2024
4,350050,3500501,Águas de Lindóia,18245.0,2024


**Nota:** o nome da aba mudou entre as publicações de 2024 (`MUNICÍPIOS`, maiúsculo) e 2025 (`Municípios`, capitalizado) — inconsistência simples de nomenclatura entre publicações do IBGE, tratada explicitamente na função.

## 3. Exploração/qualidade dos dados tratados

In [ ]:
print(f"Municípios únicos 2024: {df_pop_2024['COD_MUNIC_6'].nunique()}")
print(f"Municípios únicos 2025: {df_pop_2025['COD_MUNIC_6'].nunique()}")
print(f"Duplicatas de (COD_MUNIC_6 + ANO_REF): {df_populacao_sp.duplicated(subset=['COD_MUNIC_6', 'ANO_REF']).sum()}")
print(f"\nNulos por coluna:")
print(df_populacao_sp.isna().sum())
print(f"\nPopulação <= 0: {(df_populacao_sp['POPULACAO'] <= 0).sum()}")
print(f"Menor população: {df_populacao_sp['POPULACAO'].min()}")
print(f"Maior população: {df_populacao_sp['POPULACAO'].max()}")

Municípios únicos 2024: 645
Municípios únicos 2025: 645
Duplicatas de (COD_MUNIC_6 + ANO_REF): 0

Nulos por coluna:
COD_MUNIC_6       0
COD_MUNIC_7       0
NOME_MUNICIPIO    0
POPULACAO         0
ANO_REF           0
dtype: int64

População <= 0: 0
Menor população: 928.0
Maior população: 11904961.0


## 4. Consistência entre os dois anos disponíveis

Verificamos se o mesmo conjunto de municípios aparece nos dois anos, e se as variações de população ano a ano são plausíveis (grandes saltos poderiam indicar erro de digitação na fonte).

In [ ]:
municipios_2024 = set(df_pop_2024['COD_MUNIC_6'])
municipios_2025 = set(df_pop_2025['COD_MUNIC_6'])

print(f"Municípios só em 2024: {len(municipios_2024 - municipios_2025)}")
print(f"Municípios só em 2025: {len(municipios_2025 - municipios_2024)}")

comparacao_anos = df_pop_2024.merge(df_pop_2025, on='COD_MUNIC_6', suffixes=('_2024', '_2025'))
comparacao_anos['variacao_pct'] = (comparacao_anos['POPULACAO_2025'] - comparacao_anos['POPULACAO_2024']) / comparacao_anos['POPULACAO_2024'] * 100

print(f"\nMaior queda percentual: {comparacao_anos['variacao_pct'].min():.2f}%")
print(f"Maior crescimento percentual: {comparacao_anos['variacao_pct'].max():.2f}%")
print(f"\nMunicípios com variação > 20% (candidatos a erro de digitação):")
print(comparacao_anos[comparacao_anos['variacao_pct'].abs() > 20])

Municípios só em 2024: 0
Municípios só em 2025: 0

Maior queda percentual: -3.80%
Maior crescimento percentual: 3.50%

Municípios com variação > 20% (candidatos a erro de digitação):
Empty DataFrame
Columns: [COD_MUNIC_6, COD_MUNIC_7_2024, NOME_MUNICIPIO_2024, POPULACAO_2024, ANO_REF_2024, COD_MUNIC_7_2025, NOME_MUNICIPIO_2025, POPULACAO_2025, ANO_REF_2025, variacao_pct]
Index: []


**Resultado:** nenhuma inconsistência encontrada — mesmo conjunto de 645 municípios nos dois anos, variação máxima de -3,80% a +3,50% (plausível para dinâmica populacional real, sem indício de erro de digitação).

## 5. Cruzamento de validação com a fonte SIH/pysus (NB1)

In [ ]:
!pip install -q pysus

from pysus import sih
import pandas as pd

MESES_AMOSTRA = [(2026, m) for m in range(1, 7)]  # jan a jun/2026

dfs_amostra = []
for ano, mes in MESES_AMOSTRA:
    try:
        caminhos = sih(state="SP", year=ano, month=mes)
        caminhos_rd = [p for p in caminhos if "RDSP" in p.upper()]
        if caminhos_rd:
            dfs_amostra.append(pd.concat([pd.read_parquet(p) for p in caminhos_rd], ignore_index=True))
    except Exception as e:
        print(f'{ano}-{mes:02d} falhou: {e}')

df_sih = pd.concat(dfs_amostra, ignore_index=True)
print(f"Total de registros na amostra (SIH): {len(df_sih)}")

In [ ]:
municipios_populacao = set(df_populacao_sp['COD_MUNIC_6'])
municipios_sih_hospital = set(df_sih['MUNIC_MOV'].astype(str).str.strip())
municipios_sih_residencia = set(df_sih['MUNIC_RES'].astype(str).str.strip())

sem_pop_hospital = municipios_sih_hospital - municipios_populacao
sem_pop_residencia = municipios_sih_residencia - municipios_populacao

print(f"Municípios de HOSPITAL sem dado de população: {len(sem_pop_hospital)} de {len(municipios_sih_hospital)}")
print(f"Municípios de RESIDÊNCIA sem dado de população: {len(sem_pop_residencia)} de {len(municipios_sih_residencia)}")
print("\n(o resíduo de residência é esperado — pacientes de outros estados, mesma causa identificada no NB1)")

NameError: name 'df_sih' is not defined

**Conclusão do cruzamento:** cobertura de 100% para município de hospital — a fonte de população está pronta para uso em qualquer indicador per capita do projeto.

## Dicionário de dados — variáveis selecionadas (População IBGE)

| Coluna | Descrição |
|---|---|
| `COD_MUNIC_6` | Código IBGE de 6 dígitos (compatível com `MUNIC_MOV`/`MUNIC_RES` do SIH) |
| `COD_MUNIC_7` | Código IBGE original de 7 dígitos, com dígito verificador |
| `NOME_MUNICIPIO` | Nome do município |
| `POPULACAO` | População estimada |
| `ANO_REF` | Ano de referência da estimativa (2024 ou 2025) |

## Mini KPIs — População

In [ ]:
print("Top 10 municípios mais populosos (2025):")
print(df_pop_2025.nlargest(10, 'POPULACAO')[['NOME_MUNICIPIO', 'POPULACAO']])

print(f"\nPopulação total de SP (2025): {df_pop_2025['POPULACAO'].sum():,.0f}")
print(f"População total de SP (2024): {df_pop_2024['POPULACAO'].sum():,.0f}")
print(f"Crescimento populacional do estado: {(df_pop_2025['POPULACAO'].sum() / df_pop_2024['POPULACAO'].sum() - 1) * 100:.2f}%")
print(f"\nMediana de população por município: {df_pop_2025['POPULACAO'].median():,.0f}")
print(f"Municípios com menos de 5.000 habitantes: {(df_pop_2025['POPULACAO'] < 5000).sum()}")

Top 10 municípios mais populosos (2025):
             NOME_MUNICIPIO   POPULACAO
3829              São Paulo  11904961.0
3479              Guarulhos   1349100.0
3375               Campinas   1187974.0
3811  São Bernardo do Campo    841154.0
3801            Santo André    782048.0
3848               Sorocaba    762172.0
3654                 Osasco    759524.0
3754         Ribeirão Preto    731639.0
3824    São José dos Campos    727078.0
3823  São José do Rio Preto    504166.0

População total de SP (2025): 46,081,801
População total de SP (2024): 45,973,194
Crescimento populacional do estado: 0.24%

Mediana de população por município: 13,412
Municípios com menos de 5.000 habitantes: 148


## Conclusão da exploração — NB4

A exploração das estimativas de população do IBGE resultou em: (1) tratamento de duas particularidades estruturais da planilha (código dividido em duas colunas e inconsistência de nome de aba entre publicações); (2) zero inconsistências de qualidade (duplicata, nulo, variação anômala); (3) cobertura de 100% contra os municípios de hospital do SIH. Essa é a fonte mais simples estruturalmente entre as quatro exploradas, e está pronta para uso como base de normalização per capita (ex.: internações por 10 mil habitantes) no modelo físico a ser implementado como external table.